# Silver Layer: Data Cleaning & Validation

This notebook cleans and validates bronze data, creating silver tables.

**Tasks:**
- Clean nulls and invalid values
- Apply data quality checks
- Deduplicate records
- Standardize formats
- Write to silver Delta tables

In [ ]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, trim, lower, when, count

spark = SparkSession.builder \
    .appName("SilverTransformation") \
    .config("spark.sql.extensions", "io.delta.sql.DeltaSparkSessionExtension") \
    .config("spark.sql.catalog.spark_catalog", "org.apache.spark.sql.delta.catalog.DeltaCatalog") \
    .getOrCreate()

## Clean Customers

Transform bronze customers into silver quality

In [ ]:
# Read bronze
bronze_customers = spark.read.format("delta").load("/mnt/delta/bronze/customers")

# Clean data
silver_customers = bronze_customers \
    .withColumn("email", lower(trim(col("email")))) \
    .withColumn("city", trim(col("city"))) \
    .withColumn("state", trim(col("state"))) \
    .filter(col("customer_id").isNotNull()) \
    .filter(col("email").rlike("^\\S+@\\S+\\.\\S+$")) \
    .dropDuplicates(["customer_id"])

## Clean Products

Validate product data

In [ ]:
bronze_products = spark.read.format("delta").load("/mnt/delta/bronze/products")

silver_products = bronze_products \
    .withColumn("category", trim(col("category"))) \
    .filter((col("price") > 0) & (col("cost") > 0)) \
    .filter(col("product_id").isNotNull()) \
    .dropDuplicates(["product_id"])

## Clean Orders

Validate and enrich order data

In [ ]:
from pyspark.sql.functions import to_timestamp

bronze_orders = spark.read.format("delta").load("/mnt/delta/bronze/orders")

silver_orders = bronze_orders \
    .withColumn("order_timestamp", to_timestamp(col("order_date"))) \
    .withColumn("order_date", to_date(col("order_timestamp"))) \
    .withColumn("status", trim(col("status"))) \
    .filter(col("order_id").isNotNull()) \
    .filter(col("customer_id").isNotNull()) \
    .filter(col("total_amount") >= 0) \
    .dropDuplicates(["order_id"])

## Clean Order Items

Validate order line items

In [ ]:
bronze_order_items = spark.read.format("delta").load("/mnt/delta/bronze/order_items")

silver_order_items = bronze_order_items \
    .filter(col("order_id").isNotNull()) \
    .filter(col("product_id").isNotNull()) \
    .filter((col("quantity") > 0) & (col("unit_price") > 0)) \
    .filter(col("line_total") >= 0) \
    .withColumn("quantity", col("quantity").cast("integer")) \
    .withColumn("unit_price", col("unit_price").cast("double")) \
    .withColumn("line_total", col("line_total").cast("double")) \
    .dropDuplicates(["order_id", "product_id"])

## Write Silver Tables

In [ ]:
silver_customers.write \
    .format("delta") \
    .mode("overwrite") \
    .save("/mnt/delta/silver/customers")

silver_products.write \
    .format("delta") \
    .mode("overwrite") \
    .save("/mnt/delta/silver/products")

silver_orders.write \
    .format("delta") \
    .mode("overwrite") \
    .save("/mnt/delta/silver/orders")

silver_order_items.write \
    .format("delta") \
    .mode("overwrite") \
    .save("/mnt/delta/silver/order_items")

## Verification

In [ ]:
print("Silver tables created:")
print(f"Customers: {spark.read.format('delta').load('/mnt/delta/silver/customers').count()} rows")
print(f"Products: {spark.read.format('delta').load('/mnt/delta/silver/products').count()} rows")
print(f"Orders: {spark.read.format('delta').load('/mnt/delta/silver/orders').count()} rows")
print(f"Order Items: {spark.read.format('delta').load('/mnt/delta/silver/order_items').count()} rows")